<a href="https://colab.research.google.com/github/AldhiazTresna/UAS-KecerdasanBuatan/blob/main/uas_model_ipunb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ========================================================================
# PROYEK UAS KECERDASAN BUATAN
# PREDIKSI CURAH HUJAN DENGAN METODE ADASYN DAN LIGHT GRADIENT BOOSTING
#
# DISUSUN OLEH:
# NAMA : ALDHIAZ TRESNA LUGINA ADIESANI
# NIM  : 2406115
# ========================================================================

# ========================================================================
# SECTION 1: INSTALLASI DAN IMPORT LIBRARY
# ========================================================================

!pip install -q pandas numpy matplotlib seaborn scikit-learn lightgbm imbalanced-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import GradientBoostingClassifier
import lightgbm as lgb
from imblearn.over_sampling import ADASYN
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("Sistem Siap! Project Akhir UAS AI - Aldhiaz Tresna L. A. (2406115)")
print("="*60)

# ========================================================================
# SECTION 2: OTOMATISASI DOWNLOAD DATASET
# ========================================================================
print("\n=== DOWNLOAD DATASET DARI SERVER REPOSITORI ===")

# Mendownload dataset langsung secara otomatis agar terhindar dari FileNotFoundError
!wget -q --no-check-certificate 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pollution.csv' -O temporary_check.csv
# Link publik cadangan langsung untuk file weatherAUS.csv asli
!wget -q --no-check-certificate 'https://pub-c5e31b5cdafb419a86a69d5d34084cc5.r2.dev/weatherAUS.csv' -O weatherAUS.csv

# Memuat data ke DataFrame utama
df = pd.read_csv('weatherAUS.csv')
print("Dataset 'weatherAUS.csv' sukses diunduh dan dimuat ke sistem!")
print(f"Dimensi Data Awal: {df.shape[0]} Baris | {df.shape[1]} Fitur")

# ========================================================================
# SECTION 3: DATA PIPELINE & PENANGANAN BIAS DISTRIBUSI
# ========================================================================
print("\n=== STAGE 1: DATA PIPELINE ===")

# Reduksi dimensi dengan menghapus fitur non-kontributif dan berisiko leakage
df_clean = df.drop(['Date', 'Location', 'RISK_MM'], axis=1, errors='ignore')

# Mapping target nominal menjadi representasi boolean numerik
df_clean['RainTomorrow'] = df_clean['RainTomorrow'].map({'No': 0, 'Yes': 1})
df_clean = df_clean.dropna(subset=['RainTomorrow'])

# Encoding data tekstual/kategorikal menggunakan teknik Label Encoding
categorical_cols = df_clean.select_dtypes(include=['object']).columns.tolist()
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = df_clean[col].astype(str)
    df_clean[col] = le.fit_transform(df_clean[col])

# Rekonstruksi missing values menggunakan substitusi nilai Median sampel
for col in df_clean.columns:
    df_clean[col].fillna(df_clean[col].median(), inplace=True)

# Segmentasi variabel prediktor (X) dan variabel target (y)
X = df_clean.drop('RainTomorrow', axis=1)
y = df_clean['RainTomorrow']

# Stratified Splitting data menjadi subset Training (80%) dan Testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardisasi fitur untuk menyamakan skala rentang data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Penanganan kelangkaan sampel minoritas menggunakan algoritma ADASYN
print(f"Rasio distribusi target SEBELUM ADASYN: {np.bincount(y_train)}")
adasyn = ADASYN(random_state=42)
X_train_res, y_train_res = adasyn.fit_resample(X_train_scaled, y_train)
print(f"Rasio distribusi target SESUDAH ADASYN : {np.bincount(y_train_res)}")

# ========================================================================
# SECTION 4: ENSEMBLE CLASSIFICATION BOOSTING ARCHITECTURE
# ========================================================================
print("\n=== STAGE 2: MODEL TRAINING ===")

# Model 1: Gradient Boosting Classifier (Standard Tree Boosting)
print("Melatih Model 1: Gradient Boosting Classifier...")
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
gb_model.fit(X_train_res, y_train_res)
y_pred_gb = gb_model.predict(X_test_scaled)

# Model 2: LightGBM Classifier (Histogram-Based Leaf-Wise Boosting)
print("Melatih Model 2: LightGBM Classifier...")
lgb_model = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42, verbosity=-1)
lgb_model.fit(X_train_res, y_train_res)
y_pred_lgb = lgb_model.predict(X_test_scaled)

# ========================================================================
# SECTION 5: METRIKS EVALUASI KOMPARATIF
# ========================================================================
print("\n=== STAGE 3: METRICS COMPILATION ===")

def kalkulasi_metrik(y_true, y_pred, y_prob, model_name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)

    print(f"\n[{model_name} Evaluasi]")
    print(f"-> Akurasi  : {acc:.4f}")
    print(f"-> Presisi  : {prec:.4f}")
    print(f"-> Recall    : {rec:.4f}")
    print(f"-> F1-Score  : {f1:.4f}")
    print(f"-> ROC-AUC   : {auc:.4f}")
    return [acc, prec, rec, f1, auc]

# Perhitungan probabilitas untuk komputasi area kurva ROC
y_prob_gb = gb_model.predict_proba(X_test_scaled)[:, 1]
y_prob_lgb = lgb_model.predict_proba(X_test_scaled)[:, 1]

gb_results = kalkulasi_metrik(y_test, y_pred_gb, y_prob_gb, "Gradient Boosting")
lgb_results = kalkulasi_metrik(y_test, y_pred_lgb, y_prob_lgb, "LightGBM")

# Visualisasi Evaluasi Komparatif
metrics_names = ['Akurasi', 'Presisi', 'Recall', 'F1-Score', 'ROC-AUC']
x = np.arange(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(11, 6))
ax.bar(x - width/2, gb_results, width, label='Gradient Boosting', color='#e67e22')
ax.bar(x + width/2, lgb_results, width, label='LightGBM', color='#1abc9c')

ax.set_ylabel('Skor Performa Metrik')
ax.set_title('Grafik Analisis Komparatif Performa Model - Aldhiaz (2406115)')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend()
ax.grid(axis='y', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

# ========================================================================
# SECTION 6: KESIMPULAN AKHIR PENELITIAN
# ========================================================================
print("\n" + "="*60)
print("KESIMPULAN AKHIR PENELITIAN")
print("="*60)
best_model = "LightGBM" if lgb_results[0] > gb_results[0] else "Gradient Boosting"

print(f"1. Penanganan ketidakseimbangan dataset cuaca sukses dikoreksi lewat pendekatan ADASYN.")
print(f"2. Integrasi arsitektur berbasis Tree-Boosting modern menghasilkan performa komputasi yang efisien.")
print(f"3. Berdasarkan hasil pengujian metrik, model rekomendasi utama untuk diimplementasikan adalah: {best_model}")

print("\n" + "="*60)
print("PROYEK UAS KECERDASAN BUATAN SELESAI")
print("STUDI KASUS: METODE ENSEMBLE UNTUK KLASIFIKASI CUACA AUSTRALIA")
print("OLEH: ALDHIAZ TRESNA LUGINA ADIESANI (2406115)")
print("="*60)


Sistem Siap! Project Akhir UAS AI - Aldhiaz Tresna L. A. (2406115)

=== DOWNLOAD DATASET DARI SERVER REPOSITORI ===


EmptyDataError: No columns to parse from file